In [16]:
pip install numpy pandas matplotlib seaborn scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.


In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import confusion_matrix
from scipy.cluster.hierarchy import dendrogram, linkage

In [18]:
#Loading dataset
X = pd.read_csv("train/X_train.txt", delim_whitespace=True, header=None)
y = pd.read_csv("train/y_train.txt", header=None)

print("Feature shape:", X.shape)
print("Labels shape:", y.shape)

Feature shape: (7352, 561)
Labels shape: (7352, 1)


In [32]:
print("Missing values:", X.isnull().sum().sum())

Missing values: 0


In [33]:
#Normalize Data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [34]:
#Exploratory Data Analysis
#Feature Distribution
plt.figure(figsize=(10,5))
sns.histplot(X.iloc[:,0], bins=50)
plt.title("Feature Distribution Example")
plt.show()

In [23]:
#Dimensionality Reduction (PCA)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("PCA shape:", X_pca.shape)

PCA shape: (7352, 2)


In [24]:
#Elbow Method for K-Means
wcss = []
k_values = range(2,9)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

#Plot Elbow Curve
plt.figure(figsize=(6,4))
plt.plot(k_values, wcss, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

In [25]:
#Silhouette Curve
sil_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

#Plot
plt.figure(figsize=(6,4))
plt.plot(k_values, sil_scores, marker='o')
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs k")
plt.show()

In [26]:
#Apply K-Means
kmeans = KMeans(n_clusters=6, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

#Visualize K-Means Clusters
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=kmeans_labels, cmap='viridis')
plt.title("K-Means Clusters (PCA)")
plt.show()

In [27]:
#DBSCAN Clustering
dbscan = DBSCAN(eps=2.5, min_samples=10)
db_labels = dbscan.fit_predict(X_scaled)

#Visualize DBSCAN
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=db_labels, cmap='plasma')
plt.title("DBSCAN Clusters")
plt.show()

In [28]:
#Hierarchical Clustering
hier = AgglomerativeClustering(n_clusters=6, linkage='ward')
hier_labels = hier.fit_predict(X_scaled)

#Visualize Hierarchical Clusters
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=hier_labels, cmap='rainbow')
plt.title("Hierarchical Clustering")
plt.show()

In [29]:
#Dendrogram

sample = X_scaled[:200]
Z = linkage(sample, method='ward')
plt.figure(figsize=(10,5))
dendrogram(Z)
plt.title("Hierarchical Clustering Dendrogram")
plt.show()

In [30]:
#Evaluation Metrics
def evaluate(true_labels, pred_labels, data):
    sil = silhouette_score(data, pred_labels)
    db = davies_bouldin_score(data, pred_labels)
    ch = calinski_harabasz_score(data, pred_labels)
    ari = adjusted_rand_score(true_labels, pred_labels)
    nmi = normalized_mutual_info_score(true_labels, pred_labels)
    return sil, db, ch, ari, nmi


results = {}
results["KMeans"] = evaluate(y.values.ravel(), kmeans_labels, X_scaled)
results["DBSCAN"] = evaluate(y.values.ravel(), db_labels, X_scaled)
results["Hierarchical"] = evaluate(y.values.ravel(), hier_labels, X_scaled)
df_results = pd.DataFrame(results,index=["Silhouette",
                                 "Davies-Bouldin",
                                 "Calinski-Harabasz",
                                 "ARI",
                                 "NMI"])

print(df_results)

ValueError: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)

In [31]:
#Bar Plot Comparison
df_results.T.plot(kind='bar', figsize=(10,5))
plt.title("Clustering Algorithm Comparison")
plt.ylabel("Score")
plt.show()

#Confusion Matrix
cm = confusion_matrix(y, kmeans_labels)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix (KMeans)")
plt.show()

NameError: name 'df_results' is not defined